<a href="https://colab.research.google.com/github/DevasishK/faithful-text-summarization/blob/main/faithful_text_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#CNN- Daily Mail

In [ ]:
!pip install datasets transformers torch pandas tqdm

In [ ]:
from datasets import load_dataset
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

In [ ]:
# Load only a small subset of the TEST split
cnn = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split="test[:200]"   # start with 200 samples
)

In [ ]:
#Data Preprocessing
#Converting to DataFrame
df = pd.DataFrame(cnn)
#Removing empty / short samples
df = df[
    (df["article"].str.len() >= 200) &
    (df["highlights"].str.len() >= 50)
]
#Basic text cleaning
def clean_text(text):
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text

df["article"] = df["article"].apply(clean_text)
df["highlights"] = df["highlights"].apply(clean_text)
#Truncate article length
MAX_CHARS = 1024

df["article"] = df["article"].apply(lambda x: x[:MAX_CHARS])
#Final sanity check
print("Total samples after preprocessing:", len(df))
print(df[["article", "highlights"]].head(2))
#Save preprocessed dataset
df.to_csv("cnn_dailymail_preprocessed_200.csv", index=False)
print("Preprocessed dataset saved.")

In [ ]:
df = pd.read_csv("cnn_dailymail_preprocessed_200.csv")
print("Samples:", len(df))

#summarisation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

summarizer = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)
output = summarizer(
    df.iloc[0]["article"],
    max_new_tokens=130,
    do_sample=False
)

print(output[0]["generated_text"])

print(len(df))
print(df.iloc[0]["article"][:300])
print(output[0]["generated_text"])


In [ ]:
from tqdm import tqdm

generated_summaries = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    article = row["article"]

    output = summarizer(
        article,
        max_new_tokens=130,
        do_sample=False
    )

    generated_summaries.append(output[0]["generated_text"])

In [ ]:
df["generated_summary"] = generated_summaries
df.to_csv("cnn_dailymail_bart_summaries_200.csv", index=False)
print("CNN/DailyMail summaries saved successfully.")

In [ ]:
check_df = pd.read_csv("cnn_dailymail_bart_summaries_200.csv")
print(check_df.columns)
print(check_df.head(1))

In [ ]:
!pip install evaluate rouge-score bert-score

In [ ]:
import evaluate

df = pd.read_csv("cnn_dailymail_bart_summaries_200.csv")
print("Samples:", len(df))

#Rougue Score
rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=df["generated_summary"].tolist(),
    references=df["highlights"].tolist()
)

print("ROUGE results:")
for k, v in rouge_scores.items():
    print(f"{k}: {round(v, 4)}")


#Bert score

bertscore = evaluate.load("bertscore")

bert_scores = bertscore.compute(
    predictions=df["generated_summary"].tolist(),
    references=df["highlights"].tolist(),
    lang="en"
)

avg_f1 = sum(bert_scores["f1"]) / len(bert_scores["f1"])
print("Average BERTScore F1:", round(avg_f1, 4))

In [ ]:
metrics_df = pd.DataFrame([{
    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],
    "BERTScore-F1": avg_f1
}])

metrics_df.to_csv("cnn_dailymail_baseline_metrics_200.csv", index=False)
print("Baseline metrics saved.")

In [ ]:
!pip install transformers torch

In [ ]:
import pandas as pd

df = pd.read_csv("cnn_dailymail_bart_summaries_200.csv")
print("Samples:", len(df))

In [ ]:
from transformers import pipeline

nli_pipeline = pipeline(
    "text-classification",
    model="roberta-large-mnli",
    device=-1  # use 0 if GPU is available
)

In [ ]:
def faithfulness_score(summary, document):
    output = nli_pipeline(
        {
            "text": document,
            "text_pair": summary
        },
        truncation=True,
        max_length=512
    )

    return output["label"], output["score"]

In [ ]:
labels = []
scores = []

for _, row in df.iterrows():
    label, score = faithfulness_score(
        row["generated_summary"],
        row["article"]
    )
    labels.append(label)
    scores.append(score)

df["faithfulness_label"] = labels
df["faithfulness_score"] = scores

In [ ]:
df.to_csv("cnn_dailymail_with_faithfulness_200.csv", index=False)
print("Faithfulness results saved.")

print(df["faithfulness_label"].value_counts())

In [ ]:
#XSUM

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load XSum test split (200 samples)
xsum = load_dataset("xsum", split="test[:200]")

# Convert to DataFrame
df_xsum = pd.DataFrame(xsum)

print("Samples:", len(df_xsum))
print(df_xsum.columns)

In [ ]:
#preprocessing
def clean_text(text):
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text

# Remove very short samples
df_xsum = df_xsum[
    (df_xsum["document"].str.len() >= 200) &
    (df_xsum["summary"].str.len() >= 30)
]

# Clean text
df_xsum["document"] = df_xsum["document"].apply(clean_text)
df_xsum["summary"] = df_xsum["summary"].apply(clean_text)

# Truncate document (character-level, same as CNN/DM)
MAX_CHARS = 1024
df_xsum["document"] = df_xsum["document"].apply(lambda x: x[:MAX_CHARS])

# Save preprocessed dataset
df_xsum.to_csv("xsum_preprocessed_200.csv", index=False)
print("XSum preprocessed and saved.")

In [ ]:
#model initialisation(Bart)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

summarizer = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1  # change to 0 if GPU available
)

In [ ]:
#summary generation
from tqdm import tqdm

generated_summaries = []

for _, row in tqdm(df_xsum.iterrows(), total=len(df_xsum)):
    output = summarizer(
        row["document"],
        max_new_tokens=130,
        do_sample=False
    )
    generated_summaries.append(output[0]["generated_text"])

df_xsum["generated_summary"] = generated_summaries

In [ ]:
df_xsum.to_csv("xsum_bart_summaries_200.csv", index=False)
print("XSum summaries saved.")

In [ ]:
import evaluate
#ROUGE
rouge = evaluate.load("rouge")

rouge_scores_xsum = rouge.compute(
    predictions=df_xsum["generated_summary"].tolist(),
    references=df_xsum["summary"].tolist()
)

print("XSUM ROUGE Scores:")
for k, v in rouge_scores_xsum.items():
    print(f"{k}: {round(v, 4)}")
#BERT

bertscore = evaluate.load("bertscore")

bert_scores_xsum = bertscore.compute(
    predictions=df_xsum["generated_summary"].tolist(),
    references=df_xsum["summary"].tolist(),
    lang="en"
)

avg_bert_f1_xsum = sum(bert_scores_xsum["f1"]) / len(bert_scores_xsum["f1"])
print("XSUM Average BERTScore F1:", round(avg_bert_f1_xsum, 4))

In [ ]:
metrics_xsum = pd.DataFrame([{
    "ROUGE-1": rouge_scores_xsum["rouge1"],
    "ROUGE-2": rouge_scores_xsum["rouge2"],
    "ROUGE-L": rouge_scores_xsum["rougeL"],
    "BERTScore-F1": avg_bert_f1_xsum
}])

metrics_xsum.to_csv("xsum_baseline_metrics_200.csv", index=False)
print("XSUM baseline metrics saved.")

In [ ]:
from transformers import pipeline

nli_pipeline = pipeline(
    "text-classification",
    model="roberta-large-mnli",
    device=-1
)

def faithfulness_score(summary, document):
    output = nli_pipeline(
        {
            "text": document,
            "text_pair": summary
        },
        truncation=True,
        max_length=512
    )
    return output["label"], output["score"]

In [ ]:
labels = []
scores = []

for _, row in df_xsum.iterrows():
    label, score = faithfulness_score(
        row["generated_summary"],
        row["document"]
    )
    labels.append(label)
    scores.append(score)

df_xsum["faithfulness_label"] = labels
df_xsum["faithfulness_score"] = scores

In [ ]:
df_xsum.to_csv("xsum_with_faithfulness_200.csv", index=False)
print("XSum faithfulness results saved.")

In [ ]:
print(df_xsum["faithfulness_label"].value_counts())
print("Average faithfulness score:", df_xsum["faithfulness_score"].mean())

In [ ]:
#model initialise(pegasus)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/pegasus-xsum"

pegasus_tokenizer = AutoTokenizer.from_pretrained(model_name)
pegasus_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pegasus_model.to(device)

In [ ]:
from tqdm import tqdm

def pegasus_summarize(text, max_length=60, min_length=20):
    inputs = pegasus_tokenizer(
        text,
        truncation=True,
        padding="longest",
        return_tensors="pt"
    ).to(device)

    summary_ids = pegasus_model.generate(
        **inputs,
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        early_stopping=True
    )

    summary = pegasus_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )
    return summary

pegasus_summaries = []

for _, row in tqdm(df_xsum.iterrows(), total=len(df_xsum)):
    summary = pegasus_summarize(
        row["document"],
        max_length=60,
        min_length=20
    )
    pegasus_summaries.append(summary)

df_xsum["pegasus_summary"] = pegasus_summaries

In [ ]:
df_xsum.to_csv("xsum_200_pegasus_summaries.csv", index=False)
print("Pegasus summaries saved.")

In [ ]:
import evaluate
import pandas as pd # Import pandas here for consistency and safety

df_xsum = pd.read_csv("xsum_200_pegasus_summaries.csv") # Reload df_xsum with generated summaries

rouge = evaluate.load("rouge")

pegasus_rouge = rouge.compute(
    predictions=df_xsum["pegasus_summary"].tolist(),
    references=df_xsum["summary"].tolist()
)

print("Pegasus ROUGE:")
for k, v in pegasus_rouge.items():
    print(f"{k}: {round(v, 4)}")

# Bert score
bertscore = evaluate.load("bertscore")

pegasus_bert_scores = bertscore.compute(
    predictions=df_xsum["pegasus_summary"].tolist(),
    references=df_xsum["summary"].tolist(),
    lang="en"
)

avg_pegasus_bert_f1 = sum(pegasus_bert_scores["f1"]) / len(pegasus_bert_scores["f1"])
print("Pegasus Average BERTScore F1:", round(avg_pegasus_bert_f1, 4))

In [ ]:
def faithfulness_score(summary, document):
    output = nli_pipeline(
        {
            "text": document,
            "text_pair": summary
        },
        truncation=True,
        max_length=512
    )
    return output["label"], output["score"]

In [ ]:
labels = []
scores = []

for _, row in df_xsum.iterrows():
    label, score = faithfulness_score(
        row["pegasus_summary"],
        row["document"]
    )
    labels.append(label)
    scores.append(score)

df_xsum["pegasus_faithfulness_label"] = labels
df_xsum["pegasus_faithfulness_score"] = scores

In [ ]:
df_xsum.to_csv("xsum_200_pegasus_with_faithfulness.csv", index=False)
print("Pegasus faithfulness results saved.")

In [ ]:
pegasus_metrics = pd.DataFrame([{
    "ROUGE-1": pegasus_rouge["rouge1"],
    "ROUGE-2": pegasus_rouge["rouge2"],
    "ROUGE-L": pegasus_rouge["rougeL"],
    "Avg_Faithfulness": df_xsum["pegasus_faithfulness_score"].mean(),
    "Contradictions": (df_xsum["pegasus_faithfulness_label"] == "CONTRADICTION").sum()
}])

pegasus_metrics.to_csv("xsum_200_pegasus_metrics.csv", index=False)
print("Pegasus metrics saved.")

In [ ]:
print(df_xsum["pegasus_faithfulness_label"].value_counts())
print("Avg faithfulness:", df_xsum["pegasus_faithfulness_score"].mean())

In [ ]:
import numpy as np

df_xsum["pegasus_summary_length"] = df_xsum["pegasus_summary"].apply(
    lambda x: len(x.split())
)

In [ ]:
from scipy.stats import pearsonr, spearmanr

pearson = pearsonr(
    df_xsum["pegasus_summary_length"],
    df_xsum["pegasus_faithfulness_score"]
)

spearman = spearmanr(
    df_xsum["pegasus_summary_length"],
    df_xsum["pegasus_faithfulness_score"]
)

print("Pegasus Pearson:", pearson)
print("Pegasus Spearman:", spearman)

In [ ]:
df_xsum.groupby("pegasus_faithfulness_label")[
    "pegasus_summary_length"
].mean()

In [ ]:
#🔍 EXTRACT HALLUCINATION EXAMPLES

In [ ]:
import pandas as pd

cnn_df = pd.read_csv("cnn_dailymail_with_faithfulness_200.csv")
xsum_df = pd.read_csv("xsum_with_faithfulness_200.csv")

In [ ]:
cnn_hallucinations = cnn_df[
    cnn_df["faithfulness_label"].isin(["CONTRADICTION", "NEUTRAL"])
]

print("CNN/DailyMail hallucination samples:", len(cnn_hallucinations))

In [ ]:
xsum_hallucinations = df_xsum[
    df_xsum["faithfulness_label"].isin(["CONTRADICTION", "NEUTRAL"])
]

print("XSum hallucination samples:", len(xsum_hallucinations))

In [ ]:
cnn_examples = cnn_hallucinations.head(5)
xsum_examples = xsum_hallucinations.head(5)

cnn_examples.to_csv("cnn_hallucination_examples_200.csv", index=False)
xsum_examples.to_csv("xsum_hallucination_examples_200.csv", index=False)

print("Hallucination example files saved.")

In [ ]:
for i, row in cnn_examples.iterrows():
    print("="*80)
    print("FAITHFULNESS LABEL:", row["faithfulness_label"])
    print("\nARTICLE (snippet):\n", row["article"][:500])
    print("\nGENERATED SUMMARY:\n", row["generated_summary"])
    print("\nREFERENCE SUMMARY:\n", row["highlights"])

for i, row in xsum_examples.iterrows():
    print("="*80)
    print("FAITHFULNESS LABEL:", row["faithfulness_label"])
    print("\nARTICLE (snippet):\n", row["document"][:500])
    print("\nGENERATED SUMMARY:\n", row["generated_summary"])
    print("\nREFERENCE SUMMARY:\n", row["summary"])

In [ ]:
cnn_df["summary_length"] = cnn_df["generated_summary"].apply(
    lambda x: len(x.split())
)

In [ ]:
df_xsum["summary_length"] = df_xsum["generated_summary"].apply(
    lambda x: len(x.split())
)

In [ ]:
df_xsum["pegasus_summary_length"] = df_xsum["pegasus_summary"].apply(
    lambda x: len(x.split())
)

In [ ]:
from scipy.stats import pearsonr, spearmanr

pearson_cnn = pearsonr(
    cnn_df["summary_length"],
    cnn_df["faithfulness_score"]
)

spearman_cnn = spearmanr(
    cnn_df["summary_length"],
    cnn_df["faithfulness_score"]
)

print("CNN/BART Pearson:", pearson_cnn)
print("CNN/BART Spearman:", spearman_cnn)

cnn_df.groupby("faithfulness_label")["summary_length"].mean()

In [ ]:
pearson_xsum_bart = pearsonr(
    df_xsum["summary_length"],
    df_xsum["faithfulness_score"]
)

spearman_xsum_bart = spearmanr(
    df_xsum["summary_length"],
    df_xsum["faithfulness_score"]
)

print("XSum/BART Pearson:", pearson_xsum_bart)
print("XSum/BART Spearman:", spearman_xsum_bart)

df_xsum.groupby("faithfulness_label")["summary_length"].mean()

In [ ]:
pearson_xsum_peg = pearsonr(
    df_xsum["pegasus_summary_length"],
    df_xsum["pegasus_faithfulness_score"]
)

spearman_xsum_peg = spearmanr(
    df_xsum["pegasus_summary_length"],
    df_xsum["pegasus_faithfulness_score"]
)

print("XSum/Pegasus Pearson:", pearson_xsum_peg)
print("XSum/Pegasus Spearman:", spearman_xsum_peg)

df_xsum.groupby("pegasus_faithfulness_label")[
    "pegasus_summary_length"
].mean()

In [ ]:
length_faithfulness_results = pd.DataFrame([
    ["CNN/DailyMail", "BART", pearson_cnn[0], spearman_cnn[0]],
    ["XSum", "BART", pearson_xsum_bart[0], spearman_xsum_bart[0]],
    ["XSum", "Pegasus", pearson_xsum_peg[0], spearman_xsum_peg[0]],
], columns=["Dataset", "Model", "Pearson_r", "Spearman_r"])

length_faithfulness_results.to_csv(
    "length_vs_faithfulness_summary.csv",
    index=False
)

In [ ]:
import re

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s for s in sentences if len(s.split()) > 3]

In [ ]:
def sentence_level_faithfulness(sentences, document):
    results = []
    for sent in sentences:
        output = nli_pipeline(
            {
                "text": document,
                "text_pair": sent
            },
            truncation=True,
            max_length=512
        )
        results.append({
            "sentence": sent,
            "label": output["label"],
            "score": output["score"]
        })
    return results

In [ ]:
sentence_nli_results = []

contradictions = cnn_df[
    cnn_df["faithfulness_label"] == "CONTRADICTION"
]

for idx, row in contradictions.iterrows():
    sentences = split_sentences(row["generated_summary"])
    sent_results = sentence_level_faithfulness(
        sentences,
        row["article"]
    )
    for res in sent_results:
        sentence_nli_results.append({
            "dataset": "CNN/DailyMail",
            "model": "BART",
            "summary_id": idx,
            "sentence": res["sentence"],
            "sentence_label": res["label"],
            "sentence_score": res["score"]
        })

In [ ]:
contradictions_xsum = df_xsum[
    df_xsum["faithfulness_label"] == "CONTRADICTION"
]

for idx, row in contradictions_xsum.iterrows():
    sentences = split_sentences(row["generated_summary"])
    sent_results = sentence_level_faithfulness(
        sentences,
        row["document"]
    )
    for res in sent_results:
        sentence_nli_results.append({
            "dataset": "XSum",
            "model": "BART",
            "summary_id": idx,
            "sentence": res["sentence"],
            "sentence_label": res["label"],
            "sentence_score": res["score"]
        })

In [ ]:
peg_contradictions = df_xsum[
    df_xsum["pegasus_faithfulness_label"] == "CONTRADICTION"
]

for idx, row in peg_contradictions.iterrows():
    sentences = split_sentences(row["pegasus_summary"])
    sent_results = sentence_level_faithfulness(
        sentences,
        row["document"]
    )
    for res in sent_results:
        sentence_nli_results.append({
            "dataset": "XSum",
            "model": "Pegasus",
            "summary_id": idx,
            "sentence": res["sentence"],
            "sentence_label": res["label"],
            "sentence_score": res["score"]
        })

In [ ]:
df_sentence_nli = pd.DataFrame(sentence_nli_results)

df_sentence_nli.to_csv(
    "sentence_level_nli_contradictions.csv",
    index=False
)

print("Sentence-level NLI results saved.")

In [ ]:
df_sentence_nli["sentence_label"].value_counts()

In [ ]:
factcc_subset = df_xsum[
    df_xsum["faithfulness_label"] == "CONTRADICTION"
][["document", "generated_summary"]].copy()

print("FactCC subset size:", len(factcc_subset))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
factcc_model.to(device)
factcc_model.eval()

In [ ]:
import torch.nn.functional as F

def factcc_predict(document, summary):
    inputs = factcc_tokenizer(
        summary,
        document,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = factcc_model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)

    label_id = torch.argmax(probs).item()
    score = probs[0][label_id].item()

    label = "CONSISTENT" if label_id == 1 else "INCONSISTENT"
    return label, score

In [ ]:
factcc_labels = []
factcc_scores = []

for _, row in factcc_subset.iterrows():
    label, score = factcc_predict(
        row["document"],
        row["generated_summary"]
    )
    factcc_labels.append(label)
    factcc_scores.append(score)

factcc_subset["factcc_label"] = factcc_labels
factcc_subset["factcc_score"] = factcc_scores

In [ ]:
factcc_subset.to_csv(
    "xsum_factcc_lite_contradictions.csv",
    index=False
)

print("FactCC-lite results saved.")

In [ ]:
factcc_subset["factcc_label"].value_counts()

In [ ]:
human_subset = pd.concat([
    df_xsum[df_xsum["faithfulness_label"] == "CONTRADICTION"].sample(5, random_state=42),
    df_xsum[df_xsum["faithfulness_label"] == "NEUTRAL"].sample(4, random_state=42) # Changed sample size from 5 to 4
])

human_subset = human_subset[[
    "document",
    "generated_summary",
    "faithfulness_label"
]].reset_index(drop=True)

human_subset

In [ ]:
human_subset["human_judgment"] = ""

In [ ]:
human_subset.to_csv(
    "human_sanity_check_10_samples.csv",
    index=False
)

print("Human sanity check saved.")

In [ ]:
agreement = (
    human_subset["faithfulness_label"] ==
    human_subset["human_judgment"].replace({
        "FAITHFUL": "ENTAILMENT",
        "NOT_FAITHFUL": "CONTRADICTION"
    })
).mean()

print("Human–NLI agreement (rough):", agreement)

In [ ]:
import pandas as pd

df = pd.read_csv("xsum_with_faithfulness_200.csv")
print(df.columns)

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
)

rouge1_scores = []

for _, row in df.iterrows():
    scores = scorer.score(
        row["summary"], # Changed from "reference_summary" to "summary"
        row["generated_summary"]
    )
    rouge1_scores.append(scores["rouge1"].fmeasure)

df["rouge1"] = rouge1_scores

In [ ]:
from bert_score import score

P, R, F1 = score(
    df["generated_summary"].tolist(),
    df["summary"].tolist(),
    lang="en",
    verbose=True
)

df["bertscore_f1"] = F1.tolist()

In [ ]:
df.to_csv(
    "xsum_per_summary_metrics_200.csv",
    index=False
)

print("Per-summary metrics saved.")

In [ ]:
# XSum main file with faithfulness
df = pd.read_csv("xsum_with_faithfulness_200.csv")

# Pegasus metrics (ROUGE + BERTScore)
metrics_df = pd.read_csv("xsum_200_pegasus_metrics.csv")

# FactCC-lite (contradiction subset)
factcc_df = pd.read_csv("xsum_factcc_lite_contradictions.csv")

# Human sanity check (10 samples)
human_df = pd.read_csv("human_sanity_check_10_samples.csv")


In [ ]:
df = pd.read_csv("xsum_per_summary_metrics_200.csv")
rouge_median = df["rouge1"].median()
bert_median = df["bertscore_f1"].median()

df["rouge_bin"] = np.where(df["rouge1"] >= rouge_median, "HIGH", "LOW")
df["bert_bin"] = np.where(df["bertscore_f1"] >= bert_median, "HIGH", "LOW")

In [ ]:
df["nli_bin"] = df["faithfulness_label"].replace({
    "ENTAILMENT": "FAITHFUL",
    "CONTRADICTION": "NOT_FAITHFUL",
    "NEUTRAL": "NOT_FAITHFUL"
})

In [ ]:
factcc_df = pd.read_csv("xsum_factcc_lite_contradictions.csv")

df = df.merge(
    factcc_df[["document", "generated_summary", "factcc_label"]],
    on=["document", "generated_summary"],
    how="left"
)

df["factcc_bin"] = df["factcc_label"].replace({
    "CONSISTENT": "FAITHFUL",
    "INCONSISTENT": "NOT_FAITHFUL"
})

In [ ]:
human_df = pd.read_csv("human_sanity_check_10_samples.csv")

df = df.merge(
    human_df[["document", "generated_summary", "human_judgment"]],
    on=["document", "generated_summary"],
    how="left"
)

In [ ]:
def agreement(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return "AGREE" if a == b else "DISAGREE"

df["NLI_vs_ROUGE"]  = df.apply(lambda r: agreement(r["nli_bin"], r["rouge_bin"]), axis=1)
df["NLI_vs_BERT"]   = df.apply(lambda r: agreement(r["nli_bin"], r["bert_bin"]), axis=1)
df["NLI_vs_FactCC"] = df.apply(lambda r: agreement(r["nli_bin"], r["factcc_bin"]), axis=1)
df["NLI_vs_Human"]  = df.apply(lambda r: agreement(r["nli_bin"], r["human_judgment"]), axis=1)

In [ ]:

df["rouge_expectation"] = df["rouge_bin"].replace({
    "HIGH": "FAITHFUL",
    "LOW": "NOT_FAITHFUL"
})

df["bert_expectation"] = df["bert_bin"].replace({
    "HIGH": "FAITHFUL",
    "LOW": "NOT_FAITHFUL"
})


df["NLI_vs_ROUGE"] = df.apply(
    lambda r: agreement(r["nli_bin"], r["rouge_expectation"]), axis=1
)

df["NLI_vs_BERT"] = df.apply(
    lambda r: agreement(r["nli_bin"], r["bert_expectation"]), axis=1
)


agreement_matrix = pd.DataFrame({
    "NLI vs ROUGE": df["NLI_vs_ROUGE"].value_counts(),
    "NLI vs BERTScore": df["NLI_vs_BERT"].value_counts(),
    "NLI vs FactCC": df["NLI_vs_FactCC"].value_counts(),
    "NLI vs Human": df["NLI_vs_Human"].value_counts()
}).fillna(0).astype(int)

agreement_matrix

In [ ]:
agreement_matrix.to_csv("metric_agreement_matrix.csv")
print("metric_agreement_matrix.csv saved.")

In [ ]:
df[[
    "rouge_bin",
    "bert_bin",
    "nli_bin",
    "factcc_bin",
    "human_judgment"
]].head(10)

In [ ]:
sent_df = pd.read_csv("sentence_level_nli_contradictions.csv")
print(sent_df.columns)

In [ ]:
sent_df["is_contradiction"] = sent_df["sentence_label"].apply(
    lambda x: 1 if x == "CONTRADICTION" else 0
)

In [ ]:
localization_df = (
    sent_df
    .groupby(["dataset", "model", "summary_id"])
    .agg(
        total_sentences=("sentence", "count"),
        contradiction_sentences=("is_contradiction", "sum")
    )
    .reset_index()
)

In [ ]:
localization_df["localization_ratio"] = (
    localization_df["contradiction_sentences"] /
    localization_df["total_sentences"]
)

In [ ]:
localization_df["localization_ratio"].describe()


In [ ]:

localization_df.groupby(
    ["dataset", "model"]
)["localization_ratio"].describe()

In [ ]:
def bucket(r):
    if r == 0:
        return "No hallucination"
    elif r <= 0.25:
        return "Highly localized"
    elif r <= 0.5:
        return "Moderately localized"
    else:
        return "Diffuse"

In [ ]:
localization_df["localization_bucket"] = localization_df["localization_ratio"].apply(bucket)
localization_df.groupby(
    ["dataset", "model", "localization_bucket"]
).size().reset_index(name="count")




In [ ]:
localization_df["total_sentences"].describe()

In [ ]:
localization_df.to_csv(
    "hallucination_localization_scores.csv",
    index=False
)

print("hallucination_localization_scores.csv saved.")

In [ ]:
df = pd.read_csv("xsum_per_summary_metrics_200.csv")

In [ ]:
print(df.columns)


In [ ]:
def confidence_bucket(score):
    if score >= 0.90:
        return "High (0.90–1.00)"
    elif score >= 0.75:
        return "Medium (0.75–0.90)"
    else:
        return "Low (<0.75)"

df["nli_confidence_bucket"] = df["faithfulness_score"].apply(confidence_bucket)

In [ ]:
factcc_df = pd.read_csv("xsum_factcc_lite_contradictions.csv")
print(factcc_df.columns)

In [ ]:
df = df.merge(
    factcc_df[["document", "generated_summary", "factcc_label"]],
    on=["document", "generated_summary"],
    how="left"
)

In [ ]:
df["factcc_bin"] = df["factcc_label"].map({
    "CONSISTENT": "FAITHFUL",
    "INCONSISTENT": "NOT_FAITHFUL"
})

In [ ]:
df[["factcc_label", "factcc_bin"]].value_counts(dropna=False)

In [ ]:
df["factcc_correct"] = df["factcc_bin"].map({
    "FAITHFUL": 1,
    "NOT_FAITHFUL": 0
})

In [ ]:
human_df = pd.read_csv("human_sanity_check_10_samples.csv")
print(human_df.columns)

df = df.merge(
    human_df[["document", "generated_summary", "human_judgment"]],
    on=["document", "generated_summary"],
    how="left"
)

df["human_correct"] = df["human_judgment"].map({
    "FAITHFUL": 1,
    "NOT_FAITHFUL": 0
})

df[["human_judgment", "human_correct"]].value_counts(dropna=False)

In [ ]:
factcc_confidence_analysis = (
    df.dropna(subset=["factcc_correct"])
      .groupby("nli_confidence_bucket")["factcc_correct"]
      .agg(["count", "mean"])
      .reset_index()
      .rename(columns={"mean": "factcc_agreement_rate"})
)

factcc_confidence_analysis

In [ ]:
human_confidence_analysis = (
    df.dropna(subset=["human_correct"])
      .groupby("nli_confidence_bucket")["human_correct"]
      .agg(["count", "mean"])
      .reset_index()
      .rename(columns={"mean": "human_agreement_rate"})
)

human_confidence_analysis

In [ ]:
factcc_confidence_analysis.to_csv(
    "nli_confidence_vs_factcc_correctness.csv",
    index=False
)

human_confidence_analysis.to_csv(
    "nli_confidence_vs_human_correctness.csv",
    index=False
)

print("STEP 9 outputs saved.")

In [ ]:
bart_df = pd.read_csv("xsum_with_faithfulness_200.csv")
pegasus_df = pd.read_csv("xsum_200_pegasus_with_faithfulness.csv")

In [ ]:
print("BART columns:")
print(bart_df.columns)

print("\nPEGASUS columns:")
print(pegasus_df.columns)

In [ ]:
common_docs = (
    bart_df["document"]
    .drop_duplicates()
    .sample(n=20, random_state=42)
)

In [ ]:
bart_20 = bart_df[bart_df["document"].isin(common_docs)].copy()
pegasus_20 = pegasus_df[pegasus_df["document"].isin(common_docs)].copy()

In [ ]:
bart_20 = bart_20.rename(columns={
    "generated_summary": "bart_summary",
    "faithfulness_label": "bart_faithfulness",
    "faithfulness_score": "bart_confidence"
})

pegasus_20 = pegasus_20.rename(columns={
    "generated_summary": "pegasus_summary",
    "faithfulness_label": "pegasus_faithfulness",
    "faithfulness_score": "pegasus_confidence"
})

In [ ]:
cross_model_df = bart_20.merge(
    pegasus_20[
        ["document", "pegasus_summary", "pegasus_faithfulness", "pegasus_confidence"]
    ],
    on="document",
    how="inner"
)

In [ ]:
cross_model_df["faithfulness_same"] = (
    cross_model_df["bart_faithfulness"] ==
    cross_model_df["pegasus_faithfulness"]
)

cross_model_df["faithfulness_same"] = cross_model_df["faithfulness_same"].map({
    True: "SAME",
    False: "DIFFERENT"
})

In [ ]:
cross_model_df["faithfulness_same"].value_counts()

In [ ]:
cross_model_df.to_csv(
    "cross_model_faithfulness_stability.csv",
    index=False
)

print("cross_model_faithfulness_stability.csv saved.")

In [ ]:
cross_model_df[cross_model_df["faithfulness_same"] == "DIFFERENT"][
    ["bart_faithfulness", "pegasus_faithfulness"]
].head()

In [ ]:
df = pd.read_csv("xsum_with_faithfulness_200.csv")
loc_df = pd.read_csv("hallucination_localization_scores.csv")
sent_df = pd.read_csv("sentence_level_nli_contradictions.csv")
factcc_df = pd.read_csv("xsum_factcc_lite_contradictions.csv")

In [ ]:
contradiction_counts = (
    sent_df[sent_df["sentence_label"] == "CONTRADICTION"]
    .groupby("summary_id")
    .size()
    .reset_index(name="contradiction_count")
)

In [ ]:
# Ensure df is clean of potential leftover columns before merging
current_df_cols = df.columns.tolist()
cols_to_drop = [col for col in ['localization_ratio', 'total_sentences', 'contradiction_count'] if col in current_df_cols]
# Also check for suffixed versions if previous merge attempts left them
suffixed_cols_to_drop = [col for col in ['localization_ratio_x', 'localization_ratio_y', 'total_sentences_x', 'total_sentences_y', 'summary_id_x', 'summary_id_y'] if col in current_df_cols]
cols_to_drop.extend(suffixed_cols_to_drop)

if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

df = df.merge(
    loc_df[["summary_id", "localization_ratio", "total_sentences"]].rename(columns={'summary_id': 'id'}),
    on="id",
    how="left"
)

# Prepare contradiction_counts for merge: rename summary_id to id and explicitly select columns
contradiction_counts_for_merge = contradiction_counts.rename(columns={'summary_id': 'id'})[['id', 'contradiction_count']]
df = df.merge(
    contradiction_counts_for_merge,
    on="id",
    how="left"
)

df = df.merge(
    factcc_df[["document", "generated_summary", "factcc_label"]],
    on=["document", "generated_summary"],
    how="left"
)

df["contradiction_count"] = df["contradiction_count"].fillna(0)

In [ ]:
def assign_severity(row):
    if row["factcc_label"] == "INCONSISTENT":
        return "Severe"
    if row["contradiction_count"] >= 2:
        return "Severe"
    if row["localization_ratio"] == 1.0 and row["contradiction_count"] >= 1:
        return "Severe"
    if row["contradiction_count"] == 1:
        return "Moderate"
    if row["faithfulness_label"] == "NEUTRAL":
        return "Minor"
    return "Minor"

In [ ]:
df["severity"] = df.apply(assign_severity, axis=1)

In [ ]:
df[df["faithfulness_label"] == "CONTRADICTION"]["severity"].value_counts()

In [ ]:
metrics_df_temp = pd.read_csv("xsum_per_summary_metrics_200.csv")
df = df.merge(metrics_df_temp[['id', 'rouge1', 'bertscore_f1']], on='id', how='left')

rouge_median = df["rouge1"].median()
bert_median = df["bertscore_f1"].median()

df["rouge_bin"] = np.where(df["rouge1"] >= rouge_median, "HIGH", "LOW")
df["bert_bin"] = np.where(df["bertscore_f1"] >= bert_median, "HIGH", "LOW")

rouge_missed = df[
    (df["faithfulness_label"] == "CONTRADICTION") &
    (df["rouge_bin"] == "HIGH") &
    (df["severity"].isin(["Moderate", "Severe"]))
]

rouge_missed["severity"].value_counts()

In [ ]:
bert_missed = df[
    (df["faithfulness_label"] == "CONTRADICTION") &
    (df["bert_bin"] == "HIGH") &
    (df["severity"].isin(["Moderate", "Severe"]))
]

bert_missed["severity"].value_counts()

In [ ]:
df.to_csv(
    "hallucination_severity_analysis.csv",
    index=False
)

print("hallucination_severity_analysis.csv saved.")

In [ ]:
import re

def detect_failure_mode(row):
    summary = row["generated_summary"].lower()
    contradictions = row["contradiction_count"]

    if contradictions >= 2:
        return "Causal Error"

    if re.search(r"\b(19|20)\d{2}\b|\bjan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec", summary):
        return "Temporal Error"

    if re.search(r"\bsaid|announced|reported|confirmed\b", summary) and contradictions == 1:
        return "Over-Generalization"

    if contradictions == 1:
        return "Entity Fabrication"

    return "Minor / Unclear"

In [ ]:
df["failure_mode"] = df.apply(detect_failure_mode, axis=1)

In [ ]:
failure_mode_table = (
    df[df["faithfulness_label"] == "CONTRADICTION"]
    .copy() # Operate on a copy to avoid SettingWithCopyWarning
)

# Add 'dataset' and 'model' columns with constant values
failure_mode_table["dataset"] = "XSum"
failure_mode_table["model"] = "BART"

failure_mode_table = (
    failure_mode_table
    .groupby(["dataset", "model", "failure_mode"])
    .size()
    .reset_index(name="count")
)

failure_mode_table

In [ ]:
failure_mode_table.to_csv(
    "failure_mode_taxonomy.csv",
    index=False
)

In [ ]:
metric_profile = {
    "Fluency similarity": {
        "ROUGE": "High",
        "BERTScore": "High",
        "NLI": "Low",
        "FactCC": "Low"
    },
    "Detects severe hallucinations": {
        "ROUGE": "Low",
        "BERTScore": "Low",
        "NLI": "High",
        "FactCC": "High"
    },
    "Detects mild hallucinations": {
        "ROUGE": "Low",
        "BERTScore": "Low",
        "NLI": "Medium",
        "FactCC": "Low"
    },
    "Length robust": {
        "ROUGE": "Low",
        "BERTScore": "Low",
        "NLI": "Medium",
        "FactCC": "Medium"
    },
    "Cross-model stable (XSum)": {
        "ROUGE": "Low",
        "BERTScore": "Low",
        "NLI": "High",
        "FactCC": "Medium"
    }
}

metric_profile_df = pd.DataFrame(metric_profile).T
metric_profile_df

In [ ]:
metric_profile_df.to_csv(
    "metric_reliability_profile.csv"
)

In [ ]:
def risk_flag(row):
    if row["localization_ratio"] > 0.3:
        return "High Risk"
    if row["faithfulness_label"] == "CONTRADICTION":
        return "High Risk"
    if row["faithfulness_label"] == "NEUTRAL":
        return "Medium Risk"
    return "Low Risk"

In [ ]:
df["faithfulness_risk"] = df.apply(risk_flag, axis=1)

In [ ]:
df["faithfulness_risk"].value_counts()

In [ ]:
df.to_csv(
    "faithfulness_risk_annotated.csv",
    index=False
)

In [ ]:
# Only hallucination cases
hallucinations = df[df["faithfulness_label"] == "CONTRADICTION"]

analysis = hallucinations.groupby(
    ["rouge_bin", "bert_bin"]
).size().reset_index(name="count")

analysis

In [ ]:
analysis.to_csv("metric_conditioned_failures.csv", index=False)

In [ ]:
df["nli_conf_bucket"] = pd.cut(
    df["faithfulness_score"],
    bins=[0, 0.6, 0.75, 0.9, 1.0],
    labels=["Very Low", "Low", "Medium", "High"]
)

In [ ]:
calibration = df.groupby("nli_conf_bucket").apply(
    lambda x: (x["faithfulness_label"] == "CONTRADICTION").mean()
).reset_index(name="contradiction_rate")

calibration

In [ ]:
calibration.to_csv("nli_confidence_calibration.csv", index=False)

In [ ]:
df["hallucination_density"] = (
    df["contradiction_count"] / df["total_sentences"]
).fillna(0)

In [ ]:
# Add 'dataset' and 'model' columns to df for grouping purposes
df['dataset'] = 'XSum'
df['model'] = 'BART'

density_stats = df.groupby(["dataset", "model"])["hallucination_density"].mean()
density_stats

In [ ]:
density_stats.to_csv("hallucination_density_by_model.csv")